# 🎯 [OPS04_M04] 핵심 방어자산(지휘소·인프라) 거리 및 도달 예상 시간(TTI) 연계 다차원 위협 평가 고도화
### Google Colab 실습 환경 (작전·전투 수행 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS04/ops04_m04_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 작전·전투 수행 (Level 3) |
| **미션 ID** | OPS04_M04 |
| **미션 주제** | 핵심 방어자산(지휘소·인프라) 거리 및 도달 예상 시간(TTI) 연계 다차원 위협 평가 고도화 |
| **소요 시간** | 약 3.0시간 |
| **핵심 산출물** | `src/models/ops04_advanced_detector.py, 예외 탐지 테스트 결과` |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

## 📂 2. 표준 산출물 디렉토리 체계 및 실습 팁
- **방법 A (`%%writefile` 저장 추천):** 코드 셀 첫 줄에 `%%writefile <산출물 경로>`(예: `%%writefile src/models/ops04_baseline_model.py`)를 추가해 해당 폴더에 모듈을 저장할 수 있습니다.
- **방법 B (셀 단위 논리 대응 인정):** 별도 파일 추출 없이 노트북 셀 단위로 독립 실행·검증하고 미션 10 통합을 완료해도 정상 인정됩니다.

> **⚠️ [교육용 가상 데이터셋 및 HITL 원칙]**
> - 본 노트북의 모든 좌표·비행체 유형·수치는 보안 규정을 준수하여 **교육용 가상 데이터로 생성**됩니다.
> - 실제 무기 제원·사거리·부대 위치·군사 기밀 데이터는 일체 사용하지 않습니다.
> - 산출되는 대응 순위는 지휘관 승인을 위한 **'권고'**이며 자동 승인·자동 실행은 없습니다.

---

In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 04. 핵심 방어자산(지휘소·인프라) 거리 및 도달 예상 시간(TTI) 연계 다차원 위협 평가 고도화
**소요시간**: 약 3.0시간 | **이번 산출물**: `src/models/ops04_advanced_detector.py, 예외 탐지 테스트 결과`

### 📋 미션 개요
**방어 자산의 방호 가치, 지수 감쇠 기반 TTI, 위협 유사도 분류기를 결합해 시간 경과에 따라 위협 우선순위를 동적으로 재계산하는 다차원 평가 모델로 고도화합니다.**

- **배경 및 필요성:** 미션 3의 거리·시간 중심 Baseline은 기만체(decoy)에 의한 오경보에 취약하고 방어 자산별 중요도 차이를 반영하지 못하므로, 자산 가치와 신호·센서 특성을 통합한 다차원 판별 체계가 필요합니다.
- **핵심 과업:** `자산 가치 × exp(-TTI/300)` 식과 반사강도·신호 변동·센서 합의·민간 응답 신호 기반 위협 유사도 분류기를 결합하고, 위치 도약 스푸핑 의심 항적을 분리한 뒤 10초 경과 시 동적 순위 재계산 및 시험 상황 3종 비교를 수행합니다.
- **학습 목표 (습득 역량):** 시공간 동적 위협 점수 함수 설계 능력, 다중 특징 기반 위협 유사도 결합 및 스푸핑 이상 탐지 역량, Baseline 대비 재현율·지연시간 개선 검증 능력을 습득합니다.
- **검증 및 산출물:** 복합 위협 탐지 재현율(Recall) 85% 이상, 처리 지연 200ms 이하를 달성한 고도화 모델(`src/models/ops04_advanced_detector.py`)과 예외 탐지 테스트 결과를 산출합니다.

### 🎓 학습 목표 (습득 역량)
**[습득 역량]** 방어 자산의 방호 가치, 시간 지수 감쇠 TTI, 다중 신호 특징 기반 위협 유사도 분류기를 결합해 동적 위협 우선순위를 산출하고 기만체·스푸핑 방어 로직을 검증하는 고도화 모델링 역량을 체득합니다.

**[핵심 과업]** `자산 가치 × exp(-TTI/300)`와 반사강도·신호 변동·센서 합의·민간 응답 신호 기반 위협 유사도 분류기를 결합하고, 위치 도약 스푸핑 의심 항적을 분리한 뒤 시험 상황 3종 비교 및 10초 경과 후 순위 재계산을 검증합니다.

- **자산 가치 연계 동적 위협 평가 모델링 역량:** 단순 도달시간 대신 자산 방호 가치와 지수 감쇠 함수(`exp(-TTI/300)`)를 결합해 표적 접근에 따라 10초 단위로 우선순위가 재계산되는 동적 점수화 기법을 익힙니다.
- **다중 특징 위협 분류 및 스푸핑 방어 구현 능력:** 반사강도·신호 변동·센서 합의·민간 응답 신호를 학습한 위협 유사도 분류기로 기만체 오판을 억제하고, 속도 대비 위치 도약 항적을 격리하는 복합 방어 로직을 습득합니다.
- **Baseline 대비 다중 시나리오 비교 검증 역량:** 시험 상황 3종에서 Baseline과 재현율·정밀도·기만체 오판 건수·처리 지연(200ms 이하)을 대조해 모델 고도화의 실효성을 정량 입증하는 능력을 기릅니다.

### 📖 미션 배경 및 시나리오
#### 미션 배경 및 당위성
핵심 방어자산(지휘소, 레이더, 탄약창)의 방호 가치와 표적 도달 예상 시간(TTI)을 결합하여 복합 위협 우선순위를 동적으로 재계산하는 모델을 고도화한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **자산 방호 가치, 자산별 도달 예상시간, 위협 유사도를 결합해 우선순위를 동적으로 다시 계산하는 다차원 평가 모델로 고도화한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **자산 가치 x exp(-TTI/300)로 자산별 위협을 구하고, 반사강도·신호 변동·센서 합의·민간 응답 신호로 학습한 위협 유사도 분류기를 곱 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/ops04_advanced_detector.py 및 예외 탐지 테스트 결과` 형태의 결과물을 완성한다.

### 🤖 단계별 수행 및 AI 협업 가이드
#### Step 1. 미션 4 코드를 실행해 Baseline과 고도화 모델의 재현율·정밀도를 비교한다.
#### 이 단계의 역할
미션 4의 `advanced_rank()`를 실행해 `자산 가치 × exp(-TTI/300)`와 위협 유사도 분류기를 결합한 고도화 모델의 재현율·정밀도를 Baseline과 비교합니다.

#### 해야 할 일

- 반사강도·신호 변동·센서 합의·민간 응답 신호 4대 특징(`FEATS`)으로 학습된 `train_classifier()`와 지수 감쇠 공식의 결합 구조를 확인한다.
- AI를 Co-pilot으로 활용해 시험 상황 3종에서 Baseline과 `advanced_rank()`의 재현율·정밀도·처리 지연시간(ms)을 비교하는 로직을 점검한다.
- 고도화 모델의 평균 재현율(98.9%, 목표 85% 이상)과 처리 지연(200ms 이하)을 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "방어자산 방호 가치 * exp(-TTI/300)와 LogisticRegression 기반 위협 유사도 확률을 곱해 우선순위를 산출하는 advanced_rank()에 대해, 시험 상황 3종에서 Baseline 대비 재현율·정밀도·처리 지연(ms)을 비교 출력하는 코드를 검토해줘."

  - **AI 결과 검증 포인트:** AI가 분류기 학습 데이터와 시험 데이터를 섞어 평가하는 데이터 누수(Data Leakage)를 만들지 않았는지, 그리고 처리 지연이 200ms 이내인지 실측값으로 검증합니다.

#### 핵심 확인 지표

> 고도화 재현율 평균 98.9% (목표 85% 이상), 처리 지연 수 ms (목표 200ms 이하)

#### 학습자가 제출해야 하는 내용

- 미션 4 실행 결과 및 Baseline 대비 재현율·정밀도·지연시간 비교 로그
- AI 프롬프트 질의 및 데이터 분리·성능 검증 기록

#### Step 2. 기만·스푸핑 오판 건수 변화를 확인한다.
#### 이 단계의 역할
기만체(`decoy`) 오판 건수 감소 효과와 위치 도약 기반 스푸핑 의심 항적 격리(6/6건) 결과를 검증합니다.

#### 해야 할 일

- 고도화 모델 적용 전후 기만체 오판 건수 변화와 `jump_max` 기반 스푸핑 격리 판정 조건을 확인한다.
- AI에게 스푸핑 의심 항적 6건이 어떤 물리적 위치 도약 임계값에 의해 분리되었는지 진단 로그를 출력하도록 요청한다.
- 기만체·스푸핑 오판 0건 및 스푸핑 의심 6/6건 격리 달성 여부를 검증한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "advanced_rank()에서 기만체(decoy) 오판 건수가 Baseline 대비 몇 건 줄었는지와, 보고 속도 대비 위치 도약(jump_max)으로 스푸핑 의심 항적이 모두 격리되었는지 확인하는 예외 탐지 테스트 리포트를 출력해줘."

  - **AI 결과 검증 포인트:** 스푸핑 항적을 격리할 때 정상 고속 순항형(`fast_type`) 항적이 함께 격리되어 재현율이 떨어지지 않았는지 반드시 교차 확인합니다.

#### 핵심 확인 지표

> 기만·스푸핑 오판 0건, 스푸핑 의심 6/6건 격리

#### 학습자가 제출해야 하는 내용

- 기만·스푸핑 오판 변화표 및 예외 탐지 테스트 결과
- AI 생성 리포트 검증 및 정상 항적 오격리 여부 확인 메모

#### Step 3. 10초 경과 후 상위 5개의 도달시간과 점수가 다시 계산되는지 확인한다.
#### 이 단계의 역할
10초 경과 시 표적 이동에 따라 잔여 TTI와 다차원 위협 점수가 동적으로 재계산되어 우선순위가 갱신되는지 검증합니다.

#### 해야 할 일

- t=0초 시점과 t=10초 경과 시점의 상위 5개 표적 목록, TTI 변화량, 위협 점수 변동을 추출한다.
- AI에게 10초 전후 순위 변동이 발생한 표적의 원인(고속 표적의 급격한 TTI 감소 및 자산 가치 가중치 상승)을 요약하는 비교 함수를 요청한다.
- 동적 재계산 결과에서 급접근 표적이 상위로 정상 재배치되는지 확인한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "t=0초와 t=10초 경과 후 상위 5개 항적의 TTI, 자산 가치 점수, 위협 순위 변화를 나란히 비교하고 순위가 역전된 항적의 이유를 설명하는 출력 코드를 작성해줘."

  - **AI 결과 검증 포인트:** 10초 경과 시 속도×시간만큼 거리가 실제로 줄어 TTI가 감소했는지, 이미 방어자산을 지나쳐 멀어지는 항적의 우선순위가 적절히 낮아졌는지 수식으로 확인합니다.

#### 핵심 확인 지표

> 10초 경과 후 상위 5개 TTI·위협 점수 동적 재계산 및 순위 갱신 검증 완료

#### 학습자가 제출해야 하는 내용

- t=0s vs t=10s 상위 5개 순위 변동 비교 로그
- AI 비교 코드 및 동적 수식 검증 기록

#### Step 4. 도전 과제 및 파라미터 튜닝 최적화
#### 이 단계의 역할
`advanced_rank()`의 `jump_max` 임계값을 `2.0`에서 `1.1`로 낮췄을 때 정상 항적이 스푸핑으로 과잉 격리되는 부작용을 실험합니다.

#### 해야 할 일

- `jump_max`를 `2.0`에서 `1.1`로 낮춰 `advanced_rank()`를 재실행한다.
- AI에게 임계값 축소 시 정상 고속 항적이나 측정 오차 포함 항적이 스푸핑으로 오격리되는 현상과 재현율 하락 폭 분석을 요청한다.
- 오격리 증가 건수와 재현율 변화를 기록해 적정 `jump_max` 허용 폭의 근거를 정리한다.

#### AI 협업 가이드 (프롬프트 & 검증 지침)

  - **프롬프트 질의 예시:** "advanced_rank()의 jump_max를 2.0에서 1.1로 엄격하게 낮추었을 때 정상 항적 중 스푸핑으로 잘못 격리되는 건수와 재현율 변화를 비교 출력하고 적절한 임계값 설정 기준을 설명해줘."

  - **AI 결과 검증 포인트:** 임계값 하향 시 측정 잡음(위치오차)이 있는 정상 표적이 먼저 잘려나가는지 확인하고, 센서 위치오차 파라미터(`pos_err`)를 고려한 여유율 설정의 필요성을 검증합니다.

#### 핵심 확인 지표

> jump_max 2.0 vs 1.1 정상 항적 오격리 건수 및 재현율 변화 비교 완료

#### 학습자가 제출해야 하는 내용

- jump_max 변경 전후 비교 분석 메모 및 오격리 분석 결과
- AI 질의 프롬프트 및 센서 오차 연계 검증 기록

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정
**평가 구분:** 필수

**핵심 평가영역:** 기만/교란 공격에 대한 방어 로직의 현실성 및 신뢰성

#### 미션 PASS 조건
- 정량 기준: 복합 위협 탐지 재현율(Recall) 85% 이상, 처리 지연 200ms 이하
- 진행 기준: 복합 위협 시나리오 10종에 대한 탐지 테스트 통과
- 검증 방법: 합성 위협 데이터셋 주입 시 이상치 탐지 정확도 검증
- AI 검증 기준: 다차원 평가 모델의 학습/시험 분리, 10초 동적 재계산 정합성 및 jump_max 오격리 리스크를 AI 협업과 단위 검증으로 입증 완료
#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 AI 프롬프트 설계의 구체성, AI 산출물의 오류·보안 위반을 인간 검토(HITL)로 식별·수정한 기록, 그리고 출력 수치의 군 실무 부합성을 종합 채점한다.

In [ ]:
# [TODO : 학생 실습 4] 핵심 함수의 TODO 영역을 AI Co-pilot과 함께 구현하고 검증하세요.
"""가상 방어자산의 방호 가치와 자산별 도달 예상시간(TTI), 위협 유사도 분류기를 결합한 다차원 위협 평가 모델로 우선순위를 동적으로 재계산하고, 기만체·스푸핑 항적에 대한 방어 로직과 재현율·처리 지연을 검증한다."""
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---------------- [교육용가상자료] 가상 방어지역·가상 비행체 생성기 ----------------
# 좌표는 가상 지휘소를 원점으로 한 km 단위 평면이다. 비행체 유형은 추상 범주이며
# 속도·고도·반사강도는 학습용 임의 범위로, 실제 무기 제원·사거리·탄도와 무관하다.
ASSETS = pd.DataFrame([["A1", "CP(virtual)", 0.0, 0.0, 1.0],
                       ["A2", "Radar(virtual)", 4.0, -3.0, 0.8],
                       ["A3", "Depot(virtual)", -5.0, 4.0, 0.7],
                       ["A4", "Power(virtual)", 7.0, 6.0, 0.5]], columns=["asset", "name", "x", "y", "value"])
TYPES = {  # 유형: (위협 여부, 속도 m/s 범위, 고도 m 범위, 출현 비율)
    "uav_type": (True, (30, 60), (100, 800), 0.42),      # 저속 무인기형(가상)
    "fast_type": (True, (150, 250), (50, 200), 0.16),    # 고속 순항형(가상)
    "decoy": (False, (30, 60), (100, 800), 0.16),        # 기만체(가상)
    "civil": (False, (120, 220), (3000, 9000), 0.10),    # 민항기(가상)
    "bird": (False, (8, 18), (50, 400), 0.16),           # 조류 군집
}
TTI_URGENT = 600      # 긴급 판정 기준: 가상 자산 도달 예상시간 600초 이내
CPA_KM = 1.5          # 자산 상공 통과 판정 반경(가상)


def make_raid(seed, n=80, spoof=0):
    """가상 포화 상황 1회분(비행체 n개 + 스푸핑 가짜 항적 spoof개)을 만든다."""
    rng = np.random.default_rng(seed)
    names = list(TYPES)
    rows = []
    for i in range(n + spoof):
        typ = "spoof" if i >= n else str(rng.choice(names, p=[TYPES[k][3] for k in names]))
        threat, sp, al, _ = TYPES["decoy" if typ == "spoof" else typ]
        brg = np.radians(rng.uniform(0, 360)); dist = rng.uniform(6, 60)
        x, y = dist * np.sin(brg), dist * np.cos(brg)
        spd = rng.uniform(*sp); alt = rng.uniform(*al)
        aim = int(rng.integers(0, len(ASSETS)))
        if typ in ("uav_type", "fast_type", "decoy", "spoof"):      # 가상 자산 방향(작은 방위 오차)
            ax_, ay_ = ASSETS.loc[aim, ["x", "y"]]
            hd = np.arctan2(ax_ - x, ay_ - y) + np.radians(rng.normal(0, 0.6))
        else:
            hd = rng.uniform(0, 2 * np.pi); aim = -1
        vx, vy = spd * np.sin(hd) / 1000, spd * np.cos(hd) / 1000          # km/s
        sig = {"uav_type": ((0.2, 0.5), (0.02, 0.12), [0, .1, .5, .4]), "fast_type": ((0.1, 0.4), (0.02, 0.12), [0, .2, .4, .4]),
               "decoy": ((0.3, 0.8), (0.08, 0.35), [0, .55, .4, .05]), "spoof": ((0.3, 0.8), (0.05, 0.3), [0, .7, .3, 0]),
               "civil": ((0.7, 1.0), (0.01, 0.05), [0, 0, .1, .9]), "bird": ((0.01, 0.12), (0.1, 0.4), [0, .6, .4, 0])}[typ]
        jump = spd * rng.uniform(0.95, 1.2) if typ != "spoof" else spd * rng.uniform(4, 15)
        true_tti = np.inf
        if threat:
            d_aim = np.hypot(ASSETS.loc[aim, "x"] - x, ASSETS.loc[aim, "y"] - y)
            true_tti = d_aim * 1000 / spd
        rows.append({"id": f"T{seed % 100:02d}{i:03d}", "type": typ, "x": x, "y": y, "vx": vx, "vy": vy, "alt": alt, "spd": spd,
                     "rcs": rng.uniform(*sig[0]), "rcs_var": rng.uniform(*sig[1]), "n_sensors": int(rng.choice(4, p=sig[2])),
                     "max_jump": jump, "civ_iff": int(typ == "civil" and rng.random() < 0.95),
                     "aim": aim, "threat": threat, "true_tti": true_tti,
                     "urgent": bool(threat and true_tti <= TTI_URGENT)})
    df = pd.DataFrame(rows)
    df["true_prio"] = [ASSETS.loc[a, "value"] * np.exp(-t / 300) if th else 0.0
                       for a, t, th in zip(df["aim"], df["true_tti"], df["threat"])]
    return df


def geometry(df):
    """각 항적과 가상 자산 사이 최근접점(CPA) 거리와 도달 예상시간(TTI, s)을 계산한다."""
    px, py = df["x"].to_numpy()[:, None], df["y"].to_numpy()[:, None]
    vx, vy = df["vx"].to_numpy()[:, None], df["vy"].to_numpy()[:, None]
    rx, ry = ASSETS["x"].to_numpy()[None] - px, ASSETS["y"].to_numpy()[None] - py
    v2 = vx ** 2 + vy ** 2 + 1e-12
    t = (rx * vx + ry * vy) / v2                                  # CPA 도달 시각(s)
    cpa = np.hypot(rx - vx * t, ry - vy * t)
    tti = np.where((t > 0) & (cpa < CPA_KM), t, np.inf)
    return cpa, tti

# ---------------- 위협 점수 모델(추상 모델: 거리·도달시간·자산 가치·위협 유사도) ----------------
FEATS = ["log_spd", "alt_km", "rcs", "rcs_var", "n_sensors", "civ_iff", "jump_ratio"]


def features(df):
    return pd.DataFrame({"log_spd": np.log(df["spd"]), "alt_km": df["alt"] / 1000, "rcs": df["rcs"], "rcs_var": df["rcs_var"],
                         "n_sensors": df["n_sensors"], "civ_iff": df["civ_iff"], "jump_ratio": df["max_jump"] / df["spd"]})


def train_classifier(seeds=(101, 102, 103, 104)):
    """학습용 가상 상황(시험 상황과 다른 시드)으로 '위협 유사도' 분류기를 학습한다."""
    tr = pd.concat([make_raid(s, 120, spoof=10) for s in seeds], ignore_index=True)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    clf.fit(features(tr)[FEATS], tr["threat"].astype(int))
    return clf


def baseline_rank(df):
    """Baseline: 거리·도달시간만으로 1차 위협 점수를 매긴다(자산 가치·유형 판별 없음)."""
    _, tti = geometry(df)
    t_min = tti.min(axis=1)
    out = df[["id"]].copy()
    out["tti_s"] = t_min
    out["score"] = np.exp(-t_min / 300)
    out["urgent_pred"] = t_min <= TTI_URGENT
    return out.sort_values("score", ascending=False)


def advanced_rank(df, clf, w=None):
    """[TODO : 학생 실습 4] 자산가치 * exp(-TTI/300) * 위협유사도 결합 및 위치 도약(jump_max) 스푸핑 격리를 구현하세요."""
    # 힌트: 자산별 점수 = ASSETS["value"] * np.exp(-tti / 300), clf.predict_proba(...)[:, 1], spoof = (max_jump / spd) > jump_max
    pass  # <<< 여기에 advanced_rank() 로직을 작성한 뒤 pass를 제거하세요 >>>

clf = train_classifier()
rows = []
for seed in (7, 8, 9):                               # 시험용 가상 상황 3개(스푸핑 항적 포함)
    raid = make_raid(seed, 80, spoof=6)
    t0 = time.perf_counter()
    adv = advanced_rank(raid, clf)
    ms = (time.perf_counter() - t0) * 1000
    base = baseline_rank(raid)
    for name, r in (("Baseline", base), ("고도화", adv)):
        m = raid.merge(r[["id", "urgent_pred"]], on="id")
        y, p = m["urgent"].to_numpy(), m["urgent_pred"].to_numpy()
        rows.append({"상황": seed, "모델": name, "재현율": (p & y).sum() / y.sum(), "정밀도": (p & y).sum() / max(p.sum(), 1),
                     "기만·스푸핑 오판": int((p & m["type"].isin(["decoy", "spoof"]).to_numpy()).sum()),
                     "지연(ms)": ms if name == "고도화" else np.nan})
res = pd.DataFrame(rows)
print("[모델 비교] 실제 긴급 표적 기준")
print(res.round(3).fillna("-").to_string(index=False))

# 동적 재계산: 10초 경과 후 위치를 갱신해 순위 변화를 확인
raid = make_raid(7, 80, spoof=6)
r0 = advanced_rank(raid, clf).head(5)
moved = raid.copy(); moved["x"] += moved["vx"] * 10; moved["y"] += moved["vy"] * 10
r1 = advanced_rank(moved, clf).head(5)
print("\n[동적 재계산] 상위 5개 (현재 -> 10초 후)")
for a, b in zip(r0.itertuples(), r1.itertuples()):
    print(f"  {a.id} {a.asset} TTI {a.tti_s:5.0f}s 점수 {a.score:.2f}  ->  {b.id} {b.asset} TTI {b.tti_s:5.0f}s 점수 {b.score:.2f}")

adv_res = res[res["모델"] == "고도화"]
rec, lat = adv_res["재현율"].mean(), adv_res["지연(ms)"].max()
print(f"\n복합 위협 재현율 평균 {rec:.1%} (목표 85% 이상) -> {'달성' if rec >= 0.85 else '미달'}")
print(f"처리 지연 최대 {lat:.1f} ms (목표 200 ms 이하) -> {'달성' if lat <= 200 else '미달'}")
print(f"스푸핑 의심 항적 격리 {int(advanced_rank(raid, clf)['spoof_flag'].sum())}/6건 (운동학 불일치 = 보고 속도 대비 위치 도약 2배 초과)")
print("유의: 위협 유사도는 가상 반사강도·센서 합의·민간 응답 신호 등 추상 특징 기반이며, 최종 식별은 운용자 확인이 필요하다.")

<details>
<summary><b>💡 [참고] 미션 04 예상 실행 결과 (Output)</b></summary>

```text
[모델 비교] 실제 긴급 표적 기준
 상황       모델   재현율   정밀도  기만·스푸핑 오판 지연(ms)
  7 Baseline 1.000 0.721         11      -
  7      고도화 0.968 1.000          0  7.677
  8 Baseline 1.000 0.861          3      -
  8      고도화 1.000 1.000          0  6.501
  9 Baseline 1.000 0.703          9      -
  9      고도화 1.000 0.963          0  6.603

[동적 재계산] 상위 5개 (현재 -> 10초 후)
  T07055 A2 TTI    53s 점수 0.60  ->  T07055 A2 TTI    43s 점수 0.62
  T07014 A2 TTI   103s 점수 0.57  ->  T07014 A2 TTI    93s 점수 0.59
  T07033 A1 TTI   173s 점수 0.56  ->  T07033 A1 TTI   163s 점수 0.58
  T07004 A1 TTI   181s 점수 0.55  ->  T07004 A1 TTI   171s 점수 0.57
  T07053 A1 TTI   185s 점수 0.54  ->  T07053 A1 TTI   175s 점수 0.56

복합 위협 재현율 평균 98.9% (목표 85% 이상) -> 달성
처리 지연 최대 7.7 ms (목표 200 ms 이하) -> 달성
스푸핑 의심 항적 격리 6/6건 (운동학 불일치 = 보고 속도 대비 위치 도약 2배 초과)
유의: 위협 유사도는 가상 반사강도·센서 합의·민간 응답 신호 등 추상 특징 기반이며, 최종 식별은 운용자 확인이 필요하다.
```
</details>
